# Retinal OCT Image Classification
## Medical Image Classification using Deep Learning

**Dataset**: Kermany2018 - Retinal OCT Images

**Classes**:
- NORMAL (healthy retina)
- CNV (Choroidal Neovascularization)
- DME (Diabetic Macular Edema)
- DRUSEN (drusen deposits)

**Model**: EfficientNetB0 with Transfer Learning

## 1. Setup and Dependencies

In [ ]:
# Install required packages (run this cell if packages are not installed)
# !pip install tensorflow matplotlib seaborn scikit-learn pillow numpy pandas

In [ ]:
# Import libraries
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

# TensorFlow and Keras
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

# Scikit-learn for metrics
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.metrics import precision_recall_fscore_support

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Check TensorFlow version and GPU availability
print(f"TensorFlow version: {tf.__version__}")
print(f"GPU Available: {tf.config.list_physical_devices('GPU')}")
print(f"Num GPUs: {len(tf.config.list_physical_devices('GPU'))}")

## 1.5 CPU Optimization (For AMD GPU / Intel Xeon Systems)

**Note**: If you have an AMD GPU (like RX 5700 XT) or want to use CPU training on Intel Xeon, run this cell to enable optimizations.

In [ ]:
# CPU Optimization for Intel Xeon E5-2680 v4 (28 threads)
# Run this cell if you're using CPU training or AMD GPU

import os

# Configure threading for optimal CPU performance
CPU_THREADS = 28  # Xeon E5-2680 v4 has 14 cores, 28 threads

os.environ['OMP_NUM_THREADS'] = str(CPU_THREADS)
os.environ['TF_NUM_INTEROP_THREADS'] = '2'
os.environ['TF_NUM_INTRAOP_THREADS'] = str(CPU_THREADS)

# Intel MKL optimizations (if using intel-tensorflow)
os.environ['KMP_BLOCKTIME'] = '1'
os.environ['KMP_SETTINGS'] = '1'
os.environ['KMP_AFFINITY'] = 'granularity=fine,compact,1,0'

# Configure TensorFlow threading
tf.config.threading.set_inter_op_parallelism_threads(2)
tf.config.threading.set_intra_op_parallelism_threads(CPU_THREADS)

print("=" * 60)
print("CPU OPTIMIZATION ENABLED")
print("=" * 60)
print(f"CPU cores available: {os.cpu_count()}")
print(f"TensorFlow threads configured: {CPU_THREADS}")
print(f"Inter-op threads: 2")
print(f"Intra-op threads: {CPU_THREADS}")
print("\nRecommendations for your setup:")
print("- Install Intel TensorFlow: pip install intel-tensorflow")
print("- Or use standard TensorFlow-CPU: pip install tensorflow-cpu")
print("- Increase BATCH_SIZE to 64 (you have 64GB RAM)")
print("- Expected training time: 2-4 hours on Xeon E5-2680 v4")
print("=" * 60)

## 2. Dataset Configuration and Loading

**Note**: Download the dataset from Kaggle:
1. Go to: https://www.kaggle.com/datasets/paultimothymooney/kermany2018/data
2. Download and extract to `../data/kermany2018/`
3. Or use Kaggle API:
```bash
kaggle datasets download -d paultimothymooney/kermany2018
unzip kermany2018.zip -d ../data/kermany2018/
```

In [ ]:
# Dataset paths
BASE_DIR = '../data/kermany2018/OCT2017'  # Adjust this path based on your dataset location
TRAIN_DIR = os.path.join(BASE_DIR, 'train')
TEST_DIR = os.path.join(BASE_DIR, 'test')
VAL_DIR = os.path.join(BASE_DIR, 'val')

# Model configuration
IMG_SIZE = 224  # EfficientNetB0 input size
BATCH_SIZE = 32
EPOCHS = 30
NUM_CLASSES = 4

# Class names
CLASS_NAMES = ['CNV', 'DME', 'DRUSEN', 'NORMAL']

print(f"Train directory: {TRAIN_DIR}")
print(f"Test directory: {TEST_DIR}")
print(f"Validation directory: {VAL_DIR}")
print(f"\nImage size: {IMG_SIZE}x{IMG_SIZE}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Number of classes: {NUM_CLASSES}")

## 3. Data Exploration and Visualization

In [ ]:
# Count images in each split
def count_images(directory):
    """Count images in each class folder"""
    counts = {}
    for class_name in os.listdir(directory):
        class_path = os.path.join(directory, class_name)
        if os.path.isdir(class_path):
            counts[class_name] = len(os.listdir(class_path))
    return counts

# Count images
train_counts = count_images(TRAIN_DIR)
test_counts = count_images(TEST_DIR)
val_counts = count_images(VAL_DIR)

# Create summary DataFrame
summary_df = pd.DataFrame({
    'Train': train_counts,
    'Validation': val_counts,
    'Test': test_counts
})

print("\n=== Dataset Summary ===")
print(summary_df)
print(f"\nTotal images: {summary_df.sum().sum()}")
print(f"Train: {summary_df['Train'].sum()}")
print(f"Validation: {summary_df['Validation'].sum()}")
print(f"Test: {summary_df['Test'].sum()}")

In [ ]:
# Visualize class distribution
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for idx, (split_name, split_data) in enumerate([('Train', train_counts), 
                                                  ('Validation', val_counts), 
                                                  ('Test', test_counts)]):
    axes[idx].bar(split_data.keys(), split_data.values(), color=['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4'])
    axes[idx].set_title(f'{split_name} Set Distribution', fontsize=12, fontweight='bold')
    axes[idx].set_ylabel('Number of Images')
    axes[idx].tick_params(axis='x', rotation=45)
    
    # Add value labels on bars
    for i, (k, v) in enumerate(split_data.items()):
        axes[idx].text(i, v + max(split_data.values()) * 0.02, str(v), 
                      ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Display sample images from each class
def show_sample_images(directory, num_samples=4):
    """Display sample images from each class"""
    classes = sorted(os.listdir(directory))
    fig, axes = plt.subplots(len(classes), num_samples, figsize=(15, 12))
    
    for i, class_name in enumerate(classes):
        class_path = os.path.join(directory, class_name)
        images = os.listdir(class_path)[:num_samples]
        
        for j, img_name in enumerate(images):
            img_path = os.path.join(class_path, img_name)
            img = Image.open(img_path)
            
            axes[i, j].imshow(img, cmap='gray')
            axes[i, j].axis('off')
            
            if j == 0:
                axes[i, j].set_title(f'{class_name}\n{img.size[0]}x{img.size[1]}', 
                                    fontweight='bold', loc='left')
    
    plt.suptitle('Sample OCT Images from Each Class', fontsize=16, fontweight='bold', y=0.995)
    plt.tight_layout()
    plt.show()

show_sample_images(TRAIN_DIR, num_samples=4)

## 4. Data Preprocessing and Augmentation

In [ ]:
# Data augmentation for training
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=10,          # Rotate images up to 10 degrees
    width_shift_range=0.1,      # Shift images horizontally
    height_shift_range=0.1,     # Shift images vertically
    horizontal_flip=True,       # Flip images horizontally
    zoom_range=0.1,             # Zoom in/out
    fill_mode='nearest'
)

# Only rescaling for validation and test (no augmentation)
val_test_datagen = ImageDataGenerator(rescale=1./255)

# Create data generators
train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True,
    seed=42
)

val_generator = val_test_datagen.flow_from_directory(
    VAL_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

test_generator = val_test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

print(f"\nTraining samples: {train_generator.samples}")
print(f"Validation samples: {val_generator.samples}")
print(f"Test samples: {test_generator.samples}")
print(f"\nClass indices: {train_generator.class_indices}")

In [ ]:
# Visualize augmented images
def show_augmented_images(generator, num_images=8):
    """Display augmented images from the generator"""
    x_batch, y_batch = next(generator)
    
    fig, axes = plt.subplots(2, 4, figsize=(15, 8))
    axes = axes.ravel()
    
    for i in range(num_images):
        axes[i].imshow(x_batch[i])
        axes[i].set_title(f"Class: {CLASS_NAMES[np.argmax(y_batch[i])]}")
        axes[i].axis('off')
    
    plt.suptitle('Augmented Training Images', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()

show_augmented_images(train_generator)

## 5. Model Architecture - EfficientNetB0 with Transfer Learning

In [ ]:
def create_model(img_size=224, num_classes=4):
    """
    Create EfficientNetB0 model with custom classification head
    
    Architecture:
    - EfficientNetB0 (pre-trained on ImageNet, frozen)
    - Global Average Pooling
    - Dense layer (256 units, ReLU)
    - Dropout (0.5)
    - Output layer (4 units, Softmax)
    """
    # Load pre-trained EfficientNetB0 (without top classification layer)
    base_model = EfficientNetB0(
        include_top=False,
        weights='imagenet',
        input_shape=(img_size, img_size, 3)
    )
    
    # Freeze base model layers
    base_model.trainable = False
    
    # Create custom classification head
    inputs = layers.Input(shape=(img_size, img_size, 3))
    
    # Pass through base model
    x = base_model(inputs, training=False)
    
    # Add custom layers
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    
    # Create model
    model = models.Model(inputs=inputs, outputs=outputs)
    
    return model

# Create the model
model = create_model(IMG_SIZE, NUM_CLASSES)

# Display model architecture
print("\n=== Model Architecture ===")
model.summary()

# Count trainable parameters
trainable_params = np.sum([np.prod(v.get_shape()) for v in model.trainable_weights])
non_trainable_params = np.sum([np.prod(v.get_shape()) for v in model.non_trainable_weights])

print(f"\nTrainable parameters: {trainable_params:,}")
print(f"Non-trainable parameters: {non_trainable_params:,}")
print(f"Total parameters: {trainable_params + non_trainable_params:,}")

## 6. Model Compilation

In [ ]:
# Compile the model
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy', 
             keras.metrics.Precision(name='precision'),
             keras.metrics.Recall(name='recall')]
)

print("Model compiled successfully!")
print(f"\nOptimizer: Adam (lr=0.0001)")
print(f"Loss function: Categorical Crossentropy")
print(f"Metrics: Accuracy, Precision, Recall")

## 7. Training Callbacks Setup

In [ ]:
# Create models directory if it doesn't exist
os.makedirs('../models', exist_ok=True)

# Define callbacks
callbacks = [
    # Save best model
    ModelCheckpoint(
        '../models/best_oct_model.h5',
        monitor='val_accuracy',
        mode='max',
        save_best_only=True,
        verbose=1
    ),
    
    # Early stopping to prevent overfitting
    EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    
    # Reduce learning rate when validation loss plateaus
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-7,
        verbose=1
    )
]

print("Callbacks configured:")
print("1. ModelCheckpoint - Save best model based on validation accuracy")
print("2. EarlyStopping - Stop training if validation loss doesn't improve for 5 epochs")
print("3. ReduceLROnPlateau - Reduce learning rate by 50% if validation loss plateaus")

## 8. Model Training

In [ ]:
# Train the model
print("\n" + "="*50)
print("Starting Training...")
print("="*50 + "\n")

history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)

print("\n" + "="*50)
print("Training Completed!")
print("="*50)

## 9. Training History Visualization

In [ ]:
# Plot training history
def plot_training_history(history):
    """Plot training and validation metrics"""
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    # Accuracy
    axes[0, 0].plot(history.history['accuracy'], label='Train Accuracy', linewidth=2)
    axes[0, 0].plot(history.history['val_accuracy'], label='Val Accuracy', linewidth=2)
    axes[0, 0].set_title('Model Accuracy', fontsize=14, fontweight='bold')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Accuracy')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # Loss
    axes[0, 1].plot(history.history['loss'], label='Train Loss', linewidth=2)
    axes[0, 1].plot(history.history['val_loss'], label='Val Loss', linewidth=2)
    axes[0, 1].set_title('Model Loss', fontsize=14, fontweight='bold')
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('Loss')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    
    # Precision
    axes[1, 0].plot(history.history['precision'], label='Train Precision', linewidth=2)
    axes[1, 0].plot(history.history['val_precision'], label='Val Precision', linewidth=2)
    axes[1, 0].set_title('Model Precision', fontsize=14, fontweight='bold')
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylabel('Precision')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
    
    # Recall
    axes[1, 1].plot(history.history['recall'], label='Train Recall', linewidth=2)
    axes[1, 1].plot(history.history['val_recall'], label='Val Recall', linewidth=2)
    axes[1, 1].set_title('Model Recall', fontsize=14, fontweight='bold')
    axes[1, 1].set_xlabel('Epoch')
    axes[1, 1].set_ylabel('Recall')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

plot_training_history(history)

## 10. Model Evaluation on Test Set

In [ ]:
# Evaluate on test set
print("\n=== Evaluating on Test Set ===")
test_loss, test_accuracy, test_precision, test_recall = model.evaluate(test_generator, verbose=1)

# Calculate F1 Score
test_f1 = 2 * (test_precision * test_recall) / (test_precision + test_recall)

print(f"\n{'='*50}")
print("TEST SET RESULTS")
print(f"{'='*50}")
print(f"Test Loss:      {test_loss:.4f}")
print(f"Test Accuracy:  {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")
print(f"Test Precision: {test_precision:.4f}")
print(f"Test Recall:    {test_recall:.4f}")
print(f"Test F1-Score:  {test_f1:.4f}")
print(f"{'='*50}")

## 11. Detailed Classification Report

In [ ]:
# Generate predictions for classification report
print("Generating predictions for test set...")

# Reset generator
test_generator.reset()

# Get predictions
y_pred_probs = model.predict(test_generator, verbose=1)
y_pred = np.argmax(y_pred_probs, axis=1)

# Get true labels
y_true = test_generator.classes

# Classification report
print("\n" + "="*70)
print("DETAILED CLASSIFICATION REPORT")
print("="*70)
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

# Per-class accuracy
print("\n" + "="*50)
print("PER-CLASS METRICS")
print("="*50)
precision, recall, f1, support = precision_recall_fscore_support(y_true, y_pred)

metrics_df = pd.DataFrame({
    'Class': CLASS_NAMES,
    'Precision': precision,
    'Recall': recall,
    'F1-Score': f1,
    'Support': support
})

print(metrics_df.to_string(index=False))

## 12. Confusion Matrix

In [ ]:
# Compute confusion matrix
cm = confusion_matrix(y_true, y_pred)

# Plot confusion matrix
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=CLASS_NAMES, 
            yticklabels=CLASS_NAMES,
            cbar_kws={'label': 'Count'})
plt.title('Confusion Matrix - Test Set', fontsize=16, fontweight='bold', pad=20)
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.tight_layout()
plt.show()

# Normalized confusion matrix
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

plt.figure(figsize=(10, 8))
sns.heatmap(cm_normalized, annot=True, fmt='.2%', cmap='Blues', 
            xticklabels=CLASS_NAMES, 
            yticklabels=CLASS_NAMES,
            cbar_kws={'label': 'Percentage'})
plt.title('Normalized Confusion Matrix - Test Set', fontsize=16, fontweight='bold', pad=20)
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.tight_layout()
plt.show()

## 13. Prediction Examples with Visualization

In [ ]:
# Visualize predictions
def show_predictions(generator, model, num_images=12):
    """Display predictions with confidence scores"""
    generator.reset()
    x_batch, y_batch = next(generator)
    
    predictions = model.predict(x_batch[:num_images])
    
    fig, axes = plt.subplots(3, 4, figsize=(16, 12))
    axes = axes.ravel()
    
    for i in range(num_images):
        axes[i].imshow(x_batch[i])
        
        true_label = CLASS_NAMES[np.argmax(y_batch[i])]
        pred_label = CLASS_NAMES[np.argmax(predictions[i])]
        confidence = np.max(predictions[i]) * 100
        
        # Color code: green if correct, red if incorrect
        color = 'green' if true_label == pred_label else 'red'
        
        axes[i].set_title(
            f"True: {true_label}\nPred: {pred_label} ({confidence:.1f}%)",
            color=color,
            fontweight='bold'
        )
        axes[i].axis('off')
    
    plt.suptitle('Model Predictions on Test Images', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()

show_predictions(test_generator, model, num_images=12)

## 14. Prediction Confidence Distribution

In [ ]:
# Analyze prediction confidence
confidence_scores = np.max(y_pred_probs, axis=1)
correct_predictions = (y_pred == y_true)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Overall confidence distribution
axes[0].hist(confidence_scores, bins=50, color='skyblue', edgecolor='black', alpha=0.7)
axes[0].axvline(confidence_scores.mean(), color='red', linestyle='--', 
                linewidth=2, label=f'Mean: {confidence_scores.mean():.3f}')
axes[0].set_title('Prediction Confidence Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Confidence Score')
axes[0].set_ylabel('Frequency')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Correct vs Incorrect predictions
axes[1].hist([confidence_scores[correct_predictions], 
              confidence_scores[~correct_predictions]], 
             bins=30, label=['Correct', 'Incorrect'], 
             color=['green', 'red'], alpha=0.7, edgecolor='black')
axes[1].set_title('Confidence: Correct vs Incorrect Predictions', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Confidence Score')
axes[1].set_ylabel('Frequency')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nMean confidence (all predictions): {confidence_scores.mean():.4f}")
print(f"Mean confidence (correct): {confidence_scores[correct_predictions].mean():.4f}")
print(f"Mean confidence (incorrect): {confidence_scores[~correct_predictions].mean():.4f}")

## 15. Inference Function for New Images

In [ ]:
def predict_image(image_path, model, img_size=224):
    """
    Predict class for a single image
    
    Args:
        image_path: Path to the image file
        model: Trained Keras model
        img_size: Input image size for the model
    
    Returns:
        predicted_class: Predicted class name
        confidence: Confidence score
        all_probabilities: Probabilities for all classes
    """
    # Load and preprocess image
    img = Image.open(image_path).convert('RGB')
    img = img.resize((img_size, img_size))
    img_array = np.array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)
    
    # Make prediction
    predictions = model.predict(img_array, verbose=0)
    
    # Get results
    predicted_class_idx = np.argmax(predictions[0])
    predicted_class = CLASS_NAMES[predicted_class_idx]
    confidence = predictions[0][predicted_class_idx]
    
    return predicted_class, confidence, predictions[0]

# Test the function with a sample image
def visualize_single_prediction(image_path, model):
    """Visualize prediction for a single image"""
    predicted_class, confidence, all_probs = predict_image(image_path, model)
    
    # Create figure
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Display image
    img = Image.open(image_path)
    axes[0].imshow(img, cmap='gray')
    axes[0].set_title(f'Predicted: {predicted_class}\nConfidence: {confidence*100:.2f}%',
                     fontsize=14, fontweight='bold')
    axes[0].axis('off')
    
    # Display probability distribution
    colors = ['red' if i != np.argmax(all_probs) else 'green' for i in range(len(CLASS_NAMES))]
    axes[1].barh(CLASS_NAMES, all_probs * 100, color=colors, alpha=0.7)
    axes[1].set_xlabel('Probability (%)', fontsize=12)
    axes[1].set_title('Class Probabilities', fontsize=14, fontweight='bold')
    axes[1].grid(True, alpha=0.3, axis='x')
    
    # Add percentage labels
    for i, (cls, prob) in enumerate(zip(CLASS_NAMES, all_probs)):
        axes[1].text(prob * 100 + 1, i, f'{prob*100:.2f}%', va='center')
    
    plt.tight_layout()
    plt.show()
    
    return predicted_class, confidence

# Example usage (uncomment and use with your own image path)
# sample_image_path = os.path.join(TEST_DIR, 'NORMAL', os.listdir(os.path.join(TEST_DIR, 'NORMAL'))[0])
# pred_class, conf = visualize_single_prediction(sample_image_path, model)
# print(f"\nPrediction: {pred_class} (Confidence: {conf*100:.2f}%)")

## 16. Save Model and Export Results

In [ ]:
# Save the final model
final_model_path = '../models/final_oct_model.h5'
model.save(final_model_path)
print(f"✓ Model saved to: {final_model_path}")

# Save model in SavedModel format (TensorFlow)
tf_model_path = '../models/oct_model_savedmodel'
model.save(tf_model_path, save_format='tf')
print(f"✓ TensorFlow SavedModel saved to: {tf_model_path}")

# Save training history
history_df = pd.DataFrame(history.history)
history_csv_path = '../models/training_history.csv'
history_df.to_csv(history_csv_path, index=False)
print(f"✓ Training history saved to: {history_csv_path}")

# Save classification report
report_dict = classification_report(y_true, y_pred, target_names=CLASS_NAMES, output_dict=True)
report_df = pd.DataFrame(report_dict).transpose()
report_csv_path = '../models/classification_report.csv'
report_df.to_csv(report_csv_path)
print(f"✓ Classification report saved to: {report_csv_path}")

# Save model configuration
config = {
    'model_name': 'EfficientNetB0',
    'img_size': IMG_SIZE,
    'batch_size': BATCH_SIZE,
    'epochs_trained': len(history.history['loss']),
    'num_classes': NUM_CLASSES,
    'class_names': CLASS_NAMES,
    'test_accuracy': float(test_accuracy),
    'test_precision': float(test_precision),
    'test_recall': float(test_recall),
    'test_f1': float(test_f1)
}

import json
config_path = '../models/model_config.json'
with open(config_path, 'w') as f:
    json.dump(config, f, indent=4)
print(f"✓ Model configuration saved to: {config_path}")

print("\n" + "="*50)
print("All files saved successfully!")
print("="*50)

## 17. Load Saved Model (For Future Use)

In [ ]:
# Example: How to load the saved model in the future
"""
# Load model
from tensorflow import keras
loaded_model = keras.models.load_model('../models/best_oct_model.h5')

# Load configuration
import json
with open('../models/model_config.json', 'r') as f:
    config = json.load(f)

print(f"Model loaded successfully!")
print(f"Test Accuracy: {config['test_accuracy']*100:.2f}%")

# Use for prediction
predicted_class, confidence, probs = predict_image('path/to/image.jpg', loaded_model)
print(f"Prediction: {predicted_class} ({confidence*100:.2f}% confidence)")
"""

print("Model loading example provided above (commented out)")

## Summary

### Project Completion Checklist:

✅ **Data Loading & Exploration**
- Dataset structure analyzed
- Class distribution visualized
- Sample images displayed

✅ **Data Preprocessing**
- Image augmentation configured
- Data generators created
- Train/Val/Test splits prepared

✅ **Model Architecture**
- EfficientNetB0 with transfer learning
- Custom classification head
- Model compiled with Adam optimizer

✅ **Training**
- Model trained with callbacks
- Early stopping and LR reduction
- Best model checkpointing

✅ **Evaluation**
- Test accuracy calculated
- Classification report generated
- Confusion matrix visualized

✅ **Prediction & Inference**
- Prediction function created
- Confidence analysis performed
- Example predictions visualized

✅ **Model Saving**
- Model saved in H5 and SavedModel formats
- Training history exported
- Configuration saved

### Next Steps:

1. **Fine-tuning**: Unfreeze some layers of EfficientNetB0 for better performance
2. **Ensemble Methods**: Combine multiple models for improved accuracy
3. **Class Activation Maps**: Visualize what the model is looking at
4. **Deploy**: Create a web app or API for predictions
5. **Cross-validation**: Implement k-fold cross-validation

### References:
- Dataset: [Kermany et al. 2018](https://www.kaggle.com/datasets/paultimothymooney/kermany2018)
- EfficientNet: [Tan & Le, 2019](https://arxiv.org/abs/1905.11946)
- Transfer Learning: [TensorFlow Tutorials](https://www.tensorflow.org/tutorials/images/transfer_learning)